In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt



In [ ]:

IMAGE1_PATH = "../datasets/c1.jpg"
IMAGE2_PATH = "../datasets/c2.jpg"
RESULTS_DIR = "../results/Q3"

os.makedirs(RESULTS_DIR, exist_ok=True)

im1 = cv.imread(IMAGE1_PATH, cv.IMREAD_COLOR)
im2 = cv.imread(IMAGE2_PATH, cv.IMREAD_COLOR)

assert im1 is not None, "Could not load c1.jpg"
assert im2 is not None, "Could not load c2.jpg"

print("c1 shape:", im1.shape)
print("c2 shape:", im2.shape)

In [ ]:
im1_rgb = cv.cvtColor(im1, cv.COLOR_BGR2RGB)
im2_rgb = cv.cvtColor(im2, cv.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(im1_rgb)
ax[0].set_title("Circuit board c1")
ax[0].axis("off")

ax[1].imshow(im2_rgb)
ax[1].set_title("Circuit board c2")
ax[1].axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3_input_images.png", dpi=300)
plt.show()

### Q3(a)

In [ ]:
# Q3(a): Manual corresponding point selection using lecturer-style OpenCV windows

N = 6
n = 0

p1 = np.empty((N, 2), dtype=np.float32)
p2 = np.empty((N, 2), dtype=np.float32)

# Use reduced images for easier display and clicking
im1_click = cv.imread(IMAGE1_PATH, cv.IMREAD_REDUCED_COLOR_4)
im2_click = cv.imread(IMAGE2_PATH, cv.IMREAD_REDUCED_COLOR_4)

assert im1_click is not None, "Could not load reduced c1 image"
assert im2_click is not None, "Could not load reduced c2 image"

im1_click_copy = im1_click.copy()
im2_click_copy = im2_click.copy()

def draw_circle(event, x, y, flags, param):
    global n
    points_array = param[0]
    image_copy = param[1]
    if event == cv.EVENT_LBUTTONDOWN:
        if n < N:
            cv.circle(image_copy, (x, y), 5, (255, 0, 0), -1)
            cv.putText(
                image_copy,
                str(n + 1),
                (x + 8, y - 8),
                cv.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )
            points_array[n] = (x, y)
            print(f"Point {n + 1}: ({x}, {y})")
            n += 1

print("Click 6 points on Image 1 (c1). Press ESC if needed.")

cv.namedWindow("Image 1 - c1", cv.WINDOW_AUTOSIZE)
param = [p1, im1_click_copy]
cv.setMouseCallback("Image 1 - c1", draw_circle, param)
while True:
    cv.imshow("Image 1 - c1", im1_click_copy)
    if n >= N:
        break
    if cv.waitKey(20) & 0xFF == 27:
        break
cv.destroyWindow("Image 1 - c1")
n = 0
print("\nClick the corresponding 6 points on Image 2 (c2) in the SAME ORDER.")
cv.namedWindow("Image 2 - c2", cv.WINDOW_AUTOSIZE)
param = [p2, im2_click_copy]
cv.setMouseCallback("Image 2 - c2", draw_circle, param)
while True:
    cv.imshow("Image 2 - c2", im2_click_copy)
    
    if n >= N:
        break
    
    if cv.waitKey(20) & 0xFF == 27:
        break

cv.destroyWindow("Image 2 - c2")
cv.destroyAllWindows()

print("\np1 points from c1:")
print(p1)

print("\np2 points from c2:")
print(p2)

In [ ]:
im1_points_rgb = cv.cvtColor(im1_click_copy, cv.COLOR_BGR2RGB)
im2_points_rgb = cv.cvtColor(im2_click_copy, cv.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(im1_points_rgb)
ax[0].set_title("Manual points on c1")
ax[0].axis("off")

ax[1].imshow(im2_points_rgb)
ax[1].set_title("Corresponding manual points on c2")
ax[1].axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3a_manual_points.png", dpi=300)
plt.show()

In [ ]:
H_manual, mask_manual = cv.findHomography(p1, p2, method=cv.RANSAC)

print("Manual homography matrix:")
print(H_manual)

print("\nInlier mask:")
print(mask_manual.ravel())

warped_manual = cv.warpPerspective(
    im1_click,
    H_manual,
    (im2_click.shape[1], im2_click.shape[0])
)

warped_manual_rgb = cv.cvtColor(warped_manual, cv.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(warped_manual_rgb)
plt.title("Q3(a): c1 warped to perspective of c2 using manual homography")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3a_manual_warped.png", dpi=300)
plt.show()

In [ ]:
with open(f"{RESULTS_DIR}/q3_manual_homography.txt", "w") as f:
    f.write("Manual homography matrix H_manual:\n")
    np.savetxt(f, H_manual, fmt="%.8f")
    f.write("\n\nInlier mask:\n")
    np.savetxt(f, mask_manual.reshape(1, -1), fmt="%d")

print("Saved manual homography matrix.")

### Q3)b

In [ ]:
# Q3(b): Subtract the warped image from the other image

diff_manual = cv.absdiff(im2_click, warped_manual)
diff_manual_gray = cv.cvtColor(diff_manual, cv.COLOR_BGR2GRAY)

print("Difference image shape:", diff_manual_gray.shape)

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(diff_manual_gray, cmap="gray")
plt.title("Q3(b): Difference Image Using Manual Homography")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3b_manual_difference.png", dpi=300)
plt.show()

## Q3)c

In [ ]:
# Q3(c): Feature detection and descriptor matching using SIFT

gray1 = cv.cvtColor(im1_click, cv.COLOR_BGR2GRAY)
gray2 = cv.cvtColor(im2_click, cv.COLOR_BGR2GRAY)

print("gray1 shape:", gray1.shape)
print("gray2 shape:", gray2.shape)

In [ ]:
# Create SIFT detector
sift = cv.SIFT_create()

# Detect keypoints and compute descriptors
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print("Number of keypoints in c1:", len(kp1))
print("Number of keypoints in c2:", len(kp2))
print("Descriptor shape for c1:", des1.shape)
print("Descriptor shape for c2:", des2.shape)

In [ ]:
im1_kp = cv.drawKeypoints(
    im1_click,
    kp1,
    None,
    flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

im2_kp = cv.drawKeypoints(
    im2_click,
    kp2,
    None,
    flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(cv.cvtColor(im1_kp, cv.COLOR_BGR2RGB))
ax[0].set_title("SIFT keypoints in c1")
ax[0].axis("off")

ax[1].imshow(cv.cvtColor(im2_kp, cv.COLOR_BGR2RGB))
ax[1].set_title("SIFT keypoints in c2")
ax[1].axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3c_sift_keypoints.png", dpi=300)
plt.show()

In [ ]:
# Match SIFT descriptors using BFMatcher and Lowe's ratio test

bf = cv.BFMatcher(cv.NORM_L2)

knn_matches = bf.knnMatch(des1, des2, k=2)

good_matches = []

RATIO_THRESHOLD = 0.8

for m, n in knn_matches:
    if m.distance < RATIO_THRESHOLD * n.distance:
        good_matches.append(m)

print("Total raw knn matches:", len(knn_matches))
print("Good matches after Lowe's ratio test:", len(good_matches))

In [ ]:
# Draw top good matches for visualization

matches_to_draw = sorted(good_matches, key=lambda m: m.distance)[:50]

matched_image = cv.drawMatches(
    im1_click,
    kp1,
    im2_click,
    kp2,
    matches_to_draw,
    None,
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 8))
plt.imshow(cv.cvtColor(matched_image, cv.COLOR_BGR2RGB))
plt.title("Q3(c): SIFT Descriptor Matches")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3c_sift_matches.png", dpi=300)
plt.show()

In [ ]:
print("Total raw knn matches:", len(knn_matches))
print("Good matches after Lowe's ratio test:", len(good_matches))

## Q3)d

In [ ]:
# Q3(d): Check whether enough good matches are available for homography
MIN_MATCH_COUNT = 4

print("Number of good SIFT matches:", len(good_matches))

if len(good_matches) < MIN_MATCH_COUNT:
    raise ValueError("Not enough good matches to compute homography. Need at least 4 matches.")

In [ ]:
src_pts = np.float32([
    kp1[m.queryIdx].pt for m in good_matches
]).reshape(-1, 1, 2)

dst_pts = np.float32([
    kp2[m.trainIdx].pt for m in good_matches
]).reshape(-1, 1, 2)

print("Source points shape:", src_pts.shape)
print("Destination points shape:", dst_pts.shape)

In [ ]:
H_sift, mask_sift = cv.findHomography(
    src_pts,
    dst_pts,
    method=cv.RANSAC,
    ransacReprojThreshold=5.0
)

if H_sift is None:
    raise ValueError("Homography computation failed.")

matches_mask = mask_sift.ravel().tolist()

num_inliers = int(np.sum(mask_sift))
num_matches = len(good_matches)

print("SIFT-based homography matrix:")
print(H_sift)

print("\nNumber of good matches:", num_matches)
print("Number of RANSAC inliers:", num_inliers)
print("Inlier ratio:", num_inliers / num_matches)

In [ ]:
draw_params = dict(
    matchColor=(0, 255, 0),
    singlePointColor=None,
    matchesMask=matches_mask,
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

sift_inlier_matches_img = cv.drawMatches(
    im1_click,
    kp1,
    im2_click,
    kp2,
    good_matches,
    None,
    **draw_params
)

plt.figure(figsize=(16, 8))
plt.imshow(cv.cvtColor(sift_inlier_matches_img, cv.COLOR_BGR2RGB))
plt.title("Q3(d): SIFT Matches Accepted by RANSAC")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3d_sift_ransac_inlier_matches.png", dpi=300)
plt.show()

In [ ]:
warped_sift = cv.warpPerspective(
    im1_click,
    H_sift,
    (im2_click.shape[1], im2_click.shape[0])
)

warped_sift_rgb = cv.cvtColor(warped_sift, cv.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(warped_sift_rgb)
plt.title("Q3(d): c1 Warped to c2 Using SIFT Homography")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3d_sift_warped.png", dpi=300)
plt.show()

In [ ]:
diff_sift = cv.absdiff(im2_click, warped_sift)
diff_sift_gray = cv.cvtColor(diff_sift, cv.COLOR_BGR2GRAY)

plt.figure(figsize=(8, 8))
plt.imshow(diff_sift_gray, cmap="gray")
plt.title("Q3(d): Difference Image Using SIFT Homography")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3d_sift_difference.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 12))

ax[0, 0].imshow(cv.cvtColor(warped_manual, cv.COLOR_BGR2RGB))
ax[0, 0].set_title("Manual Homography: Warped c1")
ax[0, 0].axis("off")

ax[0, 1].imshow(cv.cvtColor(warped_sift, cv.COLOR_BGR2RGB))
ax[0, 1].set_title("SIFT Homography: Warped c1")
ax[0, 1].axis("off")

ax[1, 0].imshow(diff_manual_gray, cmap="gray")
ax[1, 0].set_title("Manual Homography: Difference")
ax[1, 0].axis("off")

ax[1, 1].imshow(diff_sift_gray, cmap="gray")
ax[1, 1].set_title("SIFT Homography: Difference")
ax[1, 1].axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/q3d_manual_vs_sift_comparison.png", dpi=300)
plt.show()

In [ ]:
with open(f"{RESULTS_DIR}/q3d_sift_homography_summary.txt", "w") as f:
    f.write("Q3(d): SIFT-based homography estimation\n\n")
    
    f.write("SIFT-based homography matrix H_sift:\n")
    np.savetxt(f, H_sift, fmt="%.8f")
    
    f.write("\n\nMatching statistics:\n")
    f.write(f"Number of good SIFT matches = {num_matches}\n")
    f.write(f"Number of RANSAC inliers = {num_inliers}\n")
    f.write(f"Inlier ratio = {num_inliers / num_matches:.4f}\n")

print("Saved Q3(d) SIFT homography summary.")